# Prueba integral en Google Colab con Tesla T4

Este notebook conserva la evidencia de una ejecución aprobada y permite repetir desde cero la instalación, carga y verificación del proyecto Text-to-SQL.

## Antes de ejecutar

1. Selecciona **Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU**.
2. Usa **Ejecutar todas**.
3. Cuando Colab solicite un archivo, sube `Proyecto-3_Grupo-1_DLYSI_Sec-30-candidato.zip`.
4. La ultima celda debe mostrar `PRUEBA COMPLETA EN T4 APROBADA`.

El ZIP ya contiene el adapter ganador; el modelo base público se descarga automáticamente desde Hugging Face.

## 1. Seleccionar T4

---

In [2]:
import torch

assert torch.cuda.is_available(), (
    "Colab no asignó una GPU. Ve a Entorno de ejecución → "
    "Cambiar tipo de entorno de ejecución → T4 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("GPU:", gpu_name)
print(f"VRAM: {gpu_memory_gb:.2f} GB")
print("CUDA de PyTorch:", torch.version.cuda)

assert "T4" in gpu_name.upper(), (
    f"Se esperaba una T4, pero Colab asignó: {gpu_name}"
)

print("GPU T4 CONFIRMADA")

GPU: Tesla T4
VRAM: 14.56 GB
CUDA de PyTorch: 13.0
GPU T4 CONFIRMADA


In [3]:
!nvidia-smi

Sun Oct 25 01:31:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---

## 2. subir y extraer el ZIP

---

In [5]:
from google.colab import files
from pathlib import Path
import zipfile
import shutil
import os

EXPECTED_ZIP = "Proyecto-3_Grupo-1_DLYSI_Sec-30-candidato.zip"

uploaded = files.upload()

assert EXPECTED_ZIP in uploaded, (
    f"Debes subir {EXPECTED_ZIP}. "
    f"Archivos recibidos: {list(uploaded)}"
)

zip_path = Path("/content") / EXPECTED_ZIP
assert zip_path.exists(), f"No se encontró {zip_path}"

destination = Path("/content/proyecto3")

if destination.exists():
    shutil.rmtree(destination)

destination.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as archive:
    damaged_file = archive.testzip()
    assert damaged_file is None, f"Archivo dañado dentro del ZIP: {damaged_file}"
    print("Archivos dentro del ZIP:", len(archive.infolist()))
    archive.extractall(destination)

solutions = list(destination.rglob("solucion.py"))

assert len(solutions) == 1, (
    f"Se esperaba exactamente un solucion.py y se encontraron {len(solutions)}: "
    f"{solutions}"
)

project_root = solutions[0].parent
os.chdir(project_root)

print("Proyecto extraído en:", project_root)
print("Directorio actual:", Path.cwd())
print("ZIP EXTRAÍDO CORRECTAMENTE")

Saving Proyecto-3_Grupo-1_DLYSI_Sec-30-candidato.zip to Proyecto-3_Grupo-1_DLYSI_Sec-30-candidato.zip
Archivos dentro del ZIP: 164
Proyecto extraído en: /content/proyecto3
Directorio actual: /content/proyecto3
ZIP EXTRAÍDO CORRECTAMENTE


---

## 3. verificar el contenido del ZIP

---

In [6]:
from pathlib import Path
import hashlib
import json

EXPECTED_SIZE = 147_770_496
EXPECTED_SHA256 = (
    "d3126183a092521cb1feffb9424b188552d27bb8fc4b51a8d13a1348d89016f1"
)

required_files = [
    "solucion.py",
    "config.json",
    "requirements.txt",
    "adapter/adapter_config.json",
    "adapter/adapter_model.safetensors",
    "datos/validacion/preguntas.jsonl",
    "resultados/validacion_propia.csv",
    "BITACORA.md",
    "CONTRIBUCIONES.md",
    "historial_git.txt",
    "presentacion.pdf",
]

missing = [path for path in required_files if not Path(path).exists()]

assert not missing, f"Faltan archivos obligatorios: {missing}"

adapter_path = Path("adapter/adapter_model.safetensors")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()

adapter_size = adapter_path.stat().st_size
adapter_hash = sha256_file(adapter_path)

print("Tamaño del adapter:", adapter_size)
print("SHA-256:", adapter_hash)

assert adapter_size == EXPECTED_SIZE, (
    "El adapter no tiene el tamaño esperado. El ZIP podría contener "
    "un puntero Git LFS en vez de los pesos reales."
)

assert adapter_hash == EXPECTED_SHA256, (
    "El SHA-256 del adapter no coincide."
)

config = json.loads(Path("config.json").read_text(encoding="utf-8"))

assert config == {
    "modelo_base": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "adapter": "adapter/",
    "usar_adapter": True,
    "semilla": 2026,
}

print("ARCHIVOS, ADAPTER Y CONFIGURACIÓN CORRECTOS")

Tamaño del adapter: 147770496
SHA-256: d3126183a092521cb1feffb9424b188552d27bb8fc4b51a8d13a1348d89016f1
ARCHIVOS, ADAPTER Y CONFIGURACIÓN CORRECTOS


---

## 3. instalar requirements.txt

---

In [7]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 162.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 33.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 132.2 MB/s  0:00:00
  Attempting uninstall: sqlglot
    Found existing installation: sqlglot 25.20.2
    Uninstalling sqlglot-25.20.2:
      Successfully uninstalled sqlglot-25.20.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
  

---

## 4. comprobar dependencias y CUDA



---

In [8]:
from pathlib import Path
import importlib.metadata as metadata
import os
import torch

solutions = list(Path("/content/proyecto3").rglob("solucion.py"))
assert len(solutions) == 1

project_root = solutions[0].parent
os.chdir(project_root)

packages = [
    "torch",
    "numpy",
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "safetensors",
    "sentencepiece",
    "sqlglot",
]

for package in packages:
    print(f"{package}: {metadata.version(package)}")

assert torch.cuda.is_available(), "PyTorch no detectó CUDA"

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

print()
print("GPU:", gpu_name)
print(f"VRAM: {gpu_memory:.3f} GB")
print("CUDA de PyTorch:", torch.version.cuda)

assert "T4" in gpu_name.upper(), (
    f"Se esperaba Tesla T4, pero se recibió: {gpu_name}"
)

print("DEPENDENCIAS Y CUDA CORRECTAS")

torch: 2.11.0+cu130
numpy: 2.1.3
transformers: 4.57.6
peft: 0.21.2
accelerate: 1.15.0
bitsandbytes: 0.50.2
safetensors: 0.8.0
sentencepiece: 0.2.2
sqlglot: 29.0.1

GPU: Tesla T4
VRAM: 14.563 GB
CUDA de PyTorch: 13.0
DEPENDENCIAS Y CUDA CORRECTAS


---

## 5. prueba integral

---

In [9]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import gc
import hashlib
import importlib.metadata as metadata
import json
import os
import time

import torch

# ------------------------------------------------------------
# 1. Localizar el proyecto, incluso después de reiniciar Colab
# ------------------------------------------------------------

solutions = list(Path("/content/proyecto3").rglob("solucion.py"))
assert len(solutions) == 1, solutions

project_root = solutions[0].parent
os.chdir(project_root)

print("Proyecto:", project_root)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Constantes verificables
# ------------------------------------------------------------

ZIP_NAME = "Proyecto-3_Grupo-1_DLYSI_Sec-30-candidato.zip"
SOURCE_COMMIT = "42edb1298ba943a6a86cab02ca2c7f9b00ab536f"

EXPECTED_ADAPTER_SIZE = 147_770_496
EXPECTED_ADAPTER_SHA256 = (
    "d3126183a092521cb1feffb9424b188552d27bb8fc4b51a8d13a1348d89016f1"
)

LOAD_LIMIT_SECONDS = 300
INFERENCE_LIMIT_SECONDS = 30
MAX_EXECUTOR_CALLS = 3

config_path = Path("config.json")
adapter_path = Path("adapter/adapter_model.safetensors")
validation_path = Path("datos/validacion/preguntas.jsonl")

# ------------------------------------------------------------
# 3. Verificar nuevamente el adapter
# ------------------------------------------------------------

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()

adapter_size = adapter_path.stat().st_size
adapter_sha256 = sha256_file(adapter_path)

assert adapter_size == EXPECTED_ADAPTER_SIZE
assert adapter_sha256 == EXPECTED_ADAPTER_SHA256

# ------------------------------------------------------------
# 4. Seleccionar un caso fácil, medio y difícil
# ------------------------------------------------------------

validation_rows = [
    json.loads(line)
    for line in validation_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

cases = []

for difficulty in ("facil", "media", "dificil"):
    case = next(
        row
        for row in validation_rows
        if row["dificultad"] == difficulty
    )
    cases.append(case)

print("Casos seleccionados:")

for case in cases:
    print(
        case["dificultad"],
        case["id"],
        case["pregunta"],
    )

# ------------------------------------------------------------
# 5. Preparar limpieza y medición
# ------------------------------------------------------------

import solucion

original_config = json.loads(
    config_path.read_text(encoding="utf-8")
)

assert original_config["usar_adapter"] is True

def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def release_runtime():
    solucion._runtime = None
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def timed_generation(schema: str, question: str):
    synchronize()
    started = time.perf_counter()

    sql = solucion.generar_sql(schema, question)

    synchronize()
    elapsed = time.perf_counter() - started

    return sql, elapsed

def valid_query_prefix(sql: str) -> bool:
    return sql.lstrip().upper().startswith(("SELECT", "WITH"))

# ------------------------------------------------------------
# 6. Probar un modo: adapter o baseline
# ------------------------------------------------------------

def test_mode(use_adapter: bool) -> dict:
    mode_name = "adapter" if use_adapter else "baseline"

    print()
    print("=" * 70)
    print("PROBANDO:", mode_name.upper())
    print("=" * 70)

    config = dict(original_config)
    config["usar_adapter"] = use_adapter

    config_path.write_text(
        json.dumps(
            config,
            ensure_ascii=False,
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

    release_runtime()
    torch.cuda.reset_peak_memory_stats()

    # Carga del modelo
    load_started = time.perf_counter()

    runtime = solucion._get_runtime()
    runtime._load()

    synchronize()
    load_seconds = time.perf_counter() - load_started

    print(f"Carga del modelo: {load_seconds:.3f} segundos")

    assert load_seconds < LOAD_LIMIT_SECONDS, (
        f"{mode_name}: la carga tomó {load_seconds:.3f}s; "
        f"el máximo es {LOAD_LIMIT_SECONDS}s."
    )

    case_results = []

    # Fácil, media y difícil; cada una se ejecuta dos veces.
    for case in cases:
        print()
        print(
            f"[{mode_name.upper()} / "
            f"{case['dificultad'].upper()}]"
        )
        print("Pregunta:", case["pregunta"])

        sql_first, first_seconds = timed_generation(
            case["esquema"],
            case["pregunta"],
        )

        sql_second, second_seconds = timed_generation(
            case["esquema"],
            case["pregunta"],
        )

        print("SQL generado:", sql_first)
        print("SQL de referencia:", case["sql"])
        print(
            f"Primera inferencia: "
            f"{first_seconds:.3f} segundos"
        )
        print(
            f"Segunda inferencia: "
            f"{second_seconds:.3f} segundos"
        )

        assert first_seconds < INFERENCE_LIMIT_SECONDS, (
            f"{mode_name}/{case['dificultad']}: "
            f"primera inferencia de {first_seconds:.3f}s."
        )

        assert second_seconds < INFERENCE_LIMIT_SECONDS, (
            f"{mode_name}/{case['dificultad']}: "
            f"segunda inferencia de {second_seconds:.3f}s."
        )

        assert sql_first == sql_second, (
            f"{mode_name}/{case['dificultad']}: "
            "la generación no fue determinista."
        )

        assert valid_query_prefix(sql_first), (
            f"{mode_name}/{case['dificultad']}: "
            "la salida no comienza con SELECT o WITH."
        )

        case_results.append(
            {
                "id": case["id"],
                "difficulty": case["dificultad"],
                "question": case["pregunta"],
                "expected_sql": case["sql"],
                "generated_sql": sql_first,
                "first_inference_seconds": round(
                    first_seconds,
                    3,
                ),
                "second_inference_seconds": round(
                    second_seconds,
                    3,
                ),
                "deterministic": sql_first == sql_second,
                "valid_query_prefix": valid_query_prefix(
                    sql_first
                ),
                "under_30_seconds": (
                    first_seconds < INFERENCE_LIMIT_SECONDS
                    and second_seconds
                    < INFERENCE_LIMIT_SECONDS
                ),
            }
        )

    # --------------------------------------------------------
    # 7. Probar ejecutar(sql) y reparación
    # --------------------------------------------------------

    repair_case = cases[1]
    executor_calls = []

    def simulated_executor(sql: str) -> dict:
        executor_calls.append(sql)

        if len(executor_calls) == 1:
            return {
                "ok": False,
                "columnas": [],
                "filas": [],
                "error": (
                    "Error simulado para comprobar "
                    "la reparación"
                ),
            }

        return {
            "ok": True,
            "columnas": ["resultado"],
            "filas": [[1]],
            "error": None,
        }

    synchronize()
    repair_started = time.perf_counter()

    repaired_sql = solucion.generar_sql(
        repair_case["esquema"],
        repair_case["pregunta"],
        simulated_executor,
    )

    synchronize()
    repair_seconds = time.perf_counter() - repair_started

    print()
    print(f"[{mode_name.upper()} / EJECUTOR]")
    print("Llamadas al ejecutor:", len(executor_calls))
    print("Tiempo total con reparación:", repair_seconds)
    print("SQL reparado:", repaired_sql)

    assert len(executor_calls) <= MAX_EXECUTOR_CALLS, (
        f"Se realizaron {len(executor_calls)} llamadas; "
        f"el máximo es {MAX_EXECUTOR_CALLS}."
    )

    assert len(executor_calls) == 2, (
        "La prueba simulada esperaba exactamente "
        "una ejecución fallida y una ejecución reparada."
    )

    assert repair_seconds < INFERENCE_LIMIT_SECONDS, (
        f"La generación y reparación tomó "
        f"{repair_seconds:.3f}s; el máximo por pregunta "
        f"es {INFERENCE_LIMIT_SECONDS}s."
    )

    assert valid_query_prefix(repaired_sql), (
        "La consulta reparada no comienza con SELECT o WITH."
    )

    peak_vram_gb = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    result = {
        "mode": mode_name,
        "usar_adapter": use_adapter,
        "load_seconds": round(load_seconds, 3),
        "load_under_300_seconds": (
            load_seconds < LOAD_LIMIT_SECONDS
        ),
        "peak_allocated_vram_gb": round(
            peak_vram_gb,
            3,
        ),
        "cases": case_results,
        "executor_test": {
            "calls": len(executor_calls),
            "under_three_calls": (
                len(executor_calls) <= MAX_EXECUTOR_CALLS
            ),
            "repair_seconds": round(
                repair_seconds,
                3,
            ),
            "under_30_seconds": (
                repair_seconds < INFERENCE_LIMIT_SECONDS
            ),
            "repaired_sql": repaired_sql,
            "valid_query_prefix": valid_query_prefix(
                repaired_sql
            ),
        },
    }

    print()
    print(mode_name.upper(), "APROBADO")

    solucion._runtime = None
    del runtime
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    return result

# ------------------------------------------------------------
# 8. Ejecutar adapter y baseline
# ------------------------------------------------------------

try:
    adapter_result = test_mode(True)
    baseline_result = test_mode(False)

finally:
    # La entrega debe conservar usar_adapter=true.
    original_config["usar_adapter"] = True

    config_path.write_text(
        json.dumps(
            original_config,
            ensure_ascii=False,
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

    release_runtime()

# ------------------------------------------------------------
# 9. Verificar que config.json fue restaurado
# ------------------------------------------------------------

restored_config = json.loads(
    config_path.read_text(encoding="utf-8")
)

assert restored_config["usar_adapter"] is True

# ------------------------------------------------------------
# 10. Generar evidencia
# ------------------------------------------------------------

package_names = [
    "torch",
    "numpy",
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "safetensors",
    "sentencepiece",
    "sqlglot",
]

package_versions = {
    package: metadata.version(package)
    for package in package_names
}

evidence = {
    "status": "passed",
    "tested_at_guatemala": datetime.now(
        ZoneInfo("America/Guatemala")
    ).isoformat(),
    "source_archive": ZIP_NAME,
    "source_commit": SOURCE_COMMIT,
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "gpu_memory_gb": round(
            torch.cuda.get_device_properties(
                0
            ).total_memory / 1024**3,
            3,
        ),
        "torch_cuda_version": torch.version.cuda,
        "package_versions": package_versions,
    },
    "adapter_artifact": {
        "path": str(adapter_path),
        "size": adapter_size,
        "sha256": adapter_sha256,
    },
    "contract_limits": {
        "model_load_seconds": LOAD_LIMIT_SECONDS,
        "inference_seconds_per_question": (
            INFERENCE_LIMIT_SECONDS
        ),
        "maximum_executor_calls": MAX_EXECUTOR_CALLS,
    },
    "adapter_test": adapter_result,
    "baseline_test": baseline_result,
    "config_restored_with_adapter": (
        restored_config["usar_adapter"] is True
    ),
}

evidence_path = Path("resultados/colab_t4.json")

evidence_path.write_text(
    json.dumps(
        evidence,
        ensure_ascii=False,
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

print()
print("=" * 70)
print("RESULTADO FINAL")
print("=" * 70)
print(json.dumps(evidence, ensure_ascii=False, indent=2))
print()
print("PRUEBA COMPLETA EN T4 APROBADA")

Proyecto: /content/proyecto3
GPU: Tesla T4
Casos seleccionados:
facil aerolinea_val-0001 Lista hasta 2 pasajeros de Guatemala, ordenados por nombre. Escenario 1.
media aerolinea_val-0008 Muestra cada pasajero y la cantidad de reservas con estado confirmed, incluyendo quienes no tienen. Escenario 1.
dificil aerolinea_val-0015 Lista reservas cuyo total supera el promedio de su mismo estado confirmed desde 2024-01-01. Escenario 1.

PROBANDO: ADAPTER


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Carga del modelo: 42.728 segundos

[ADAPTER / FACIL]
Pregunta: Lista hasta 2 pasajeros de Guatemala, ordenados por nombre. Escenario 1.
SQL generado: SELECT `passenger_name` FROM `passengers` WHERE `country` = 'Guatemala' ORDER BY `passenger_name` LIMIT 2;
SQL de referencia: SELECT `passenger_name` FROM `passengers` WHERE `country` = 'Guatemala' ORDER BY `passenger_name` LIMIT 2;
Primera inferencia: 5.079 segundos
Segunda inferencia: 3.442 segundos

[ADAPTER / MEDIA]
Pregunta: Muestra cada pasajero y la cantidad de reservas con estado confirmed, incluyendo quienes no tienen. Escenario 1.
SQL generado: SELECT m.`passenger_name`, COUNT(e.`booking_id`) AS cantidad FROM `passengers` AS m LEFT JOIN `bookings` AS e ON e.`passenger_id` = m.`passenger_id` AND e.`status` = 'confirmed' GROUP BY m.`passenger_id`, m.`passenger_name` ORDER BY cantidad DESC;
SQL de referencia: SELECT m.`passenger_name`, COUNT(e.`booking_id`) AS cantidad FROM `passengers` AS m LEFT JOIN `bookings` AS e ON e.`passenge

---

## 6. descargar la evidencia

---

In [10]:
from google.colab import files
from pathlib import Path

evidence_path = Path(
    "/content/proyecto3/resultados/colab_t4.json"
)

if not evidence_path.exists():
    matches = list(
        Path("/content/proyecto3").rglob(
            "colab_t4.json"
        )
    )
    assert len(matches) == 1, matches
    evidence_path = matches[0]

print("Descargando:", evidence_path)

files.download(str(evidence_path))

Descargando: /content/proyecto3/resultados/colab_t4.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

---